# Portfolio edition
Nicholas Koch | MLB home run probabilities

This copy preserves the uploaded model code and clears saved outputs and execution metadata. Use Google Colab: the included storage cell requires Colab and Google Drive. Python syntax was checked during portfolio preparation; external-data downloads, training, and forecast performance were not independently rerun. See the project summary for methods and limitations.


# Home Run Model
### Daily probabilities from hitter quality, recent form and matchups

**Goal:** rank hitters by their chance of hitting **at least one home run in a game**. A 20% prediction means that, across many comparable predictions, about one in five should result in a home run. It does not mean the model expects one home run every five games for that specific player.

**Status — September 18, 2026:** this is an implemented first version, not a trained or validated forecasting product. The MLB schedule and lineup parser was checked on a real historical slate. The feature and model pipeline passed tests with 13,140 simulated hitter/game records, including checks that future results cannot change earlier inputs. Statcast downloads returned HTTP 403 in the development environment, so **no real historical performance or real player probabilities are claimed here**. This notebook contains no invented model results. Run it in an environment that can access Baseball Savant, such as your Colab or Jupyter setup; that access is not guaranteed.

**Start here:** upload this notebook to Google Colab or open it in Jupyter, run the installation cell, set `TARGET_DATE`, then run the remaining cells from top to bottom. The first run downloads several seasons and can take a while. Later runs reuse the cached data. Keep the cache if you want to avoid downloading it again.

The early-board section ranks every active non-pitcher conditional on starting; the confirmed-board section uses posted lineups. The preset builds history from 2022, trains on 2023–2024, and reserves separate parts of 2025 for model selection, calibration and testing. Daily predictions use updated data through yesterday. The daily section requires a date in 2026 or later and enough recent games to calibrate. An early-season version needs a longer calibration window.

## What goes into the model

| Part | Inputs in this version | Why it is included |
|---|---|---|
| Hitter's power | Prior 365-day HR/PA, barrel rate, hard-hit rate, exit velocity, fly-ball rate, strikeout rate and expected wOBA on contact | Establishes the hitter's underlying power and contact profile. |
| Recent performance | The same metrics over the prior 30 and 14 calendar days; recent changes in contact quality | Lets recent form matter without trusting a tiny hot streak too much. |
| Opposing starter | HR, barrels, hard contact, strikeouts and fly balls allowed; overall and handedness splits | Describes the starter's history of allowing damaging contact. |
| Pitch matchup | Starter's recent pitch mix against the hitter's side, combined with the hitter's contact results against those pitches and that pitcher handedness | Measures whether the hitter's strengths fit the starter's repertoire. |
| Bullpen | Opponent team's prior 60-day relief-pitching results | A hitter also faces relievers during the game. |
| Opportunity and environment | Confirmed batting order, or an estimate from prior starts for the early board; home/away, day/night, month, and venue interacted with batting side | Accounts for different opportunities and park environments. |

Hard-hit balls use the 95 mph threshold. Barrels use Statcast's supplied classification. Contact rates use the available measured batted balls; a missing measurement is **not** treated as weak contact. `xwobacon` is expected wOBA on contact, not an expected home-run probability or xSLG.

The model learns a venue effect. It does **not** import an official HR park-factor number. Weather, roof status, pull direction and pitch movement/location matchups are follow-up additions.

Data references: [Statcast CSV field definitions](https://baseballsavant.mlb.com/csv-docs), [pybaseball Statcast documentation](https://github.com/jldbc/pybaseball/blob/master/docs/statcast.md), and [MLB's public schedule feed](https://statsapi.mlb.com/api/v1/schedule?sportId=1&date=2025-09-18&hydrate=probablePitcher,lineups,team).

## 1. Install the packages

Run this once per new environment. If Colab asks you to restart the session after installation, restart and continue from the imports cell. Core calculations were checked with pandas 2.2.3 and scikit-learn 1.8.0. `pybaseball` supplies the download function and depends on Baseball Savant availability.

In [ ]:
%pip install -q pybaseball "pandas>=2.2,<3" "scikit-learn>=1.5,<2" matplotlib

## 2. Load the tools

`pandas` organizes the data, `numpy` handles calculations, and scikit-learn fits the model. The rest mostly downloads data and saves results. All of the model code is included below; there is no separate helper file to download.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.request import urlopen
from urllib.parse import urlencode
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Use one mount point that is never used as a cache before Drive is mounted.
DRIVE_ROOT = Path("/content/actual_gdrive")
drive.mount(str(DRIVE_ROOT))

MODEL_DIR = DRIVE_ROOT / "MyDrive" / "Home_Run_Model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Rescue files that may have been written into an unmounted placeholder folder.
# Existing permanent files are never overwritten.
for old_root in [Path("/content/drive"), Path("/content/gdrive")]:
    old_model_dir = old_root / "MyDrive" / "Home_Run_Model"

    if not old_model_dir.exists():
        continue

    copied = 0

    for old_file in old_model_dir.rglob("*"):
        if not old_file.is_file():
            continue

        relative_path = old_file.relative_to(old_model_dir)
        new_file = MODEL_DIR / relative_path

        if not new_file.exists():
            new_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old_file, new_file)
            copied += 1

    if copied:
        print(f"Recovered {copied} files from {old_model_dir}.")

print("Permanent model folder:", MODEL_DIR)


## 3. Choose the prediction date

`TARGET_DATE` defaults to today's baseball date in Eastern Time. You can replace it with a string such as `"2026-09-18"`. Historical statistics stop at the previous day.

The nine pitch groups separate four-seamers, sinkers, cutters, sliders, sweepers, curves, changeups, splitters and other pitches. Knuckle curves and slow curves are grouped with curves. These groupings are modeling choices, not proof that pitches within each group are interchangeable.

In [ ]:
# Automatically use today's date in the Eastern time zone.
# Replace this with "YYYY-MM-DD" only when you intentionally want another date.
TARGET_DATE = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d")

HISTORY_START = "2022-01-01"     # 2022 supplies history before training begins.
TRAIN_START = "2023-01-01"
VALID_START = "2025-01-01"       # Choose features and regularization here.
CALIBRATION_START = "2025-07-01" # Adjust probability levels on separate games.
TEST_START = "2025-09-01"        # Evaluate once on Sep./Oct. 2025 games.
TEST_END = "2025-12-31"

CACHE = MODEL_DIR / "hr_model_cache"
CACHE.mkdir(parents=True, exist_ok=True)

print("Target date:", TARGET_DATE)
print("Cache folder:", CACHE)
print("Saved Statcast blocks:", len(list(CACHE.glob("pa_*.csv.gz"))))

PITCHES = ["FF", "SI", "FC", "SL", "ST", "CU", "CH", "FS", "OTHER"]
PITCH_NAMES = {"FF": "four-seam", "SI": "sinker", "FC": "cutter",
               "SL": "slider", "ST": "sweeper", "CU": "curveball",
               "CH": "changeup", "FS": "splitter", "OTHER": "other"}
PA_EVENTS = {"single", "double", "triple", "home_run", "walk",
             "intent_walk", "intentional_walk", "hit_by_pitch", "strikeout",
             "strikeout_double_play", "field_out", "force_out", "field_error",
             "fielders_choice", "fielders_choice_out", "grounded_into_double_play",
             "double_play", "triple_play", "sac_fly", "sac_bunt",
             "sac_fly_double_play", "sac_bunt_double_play", "catcher_interf"}


## 4. Get games, starting lineups and handedness

One observation will be **one starting hitter in one game**. MLB player IDs keep players with similar names separate. A switch hitter's batting side is set using the opposing starter's throwing hand.

The early board needs an active-roster hitter and a named opposing starter. It reports P(HR | starts), without estimating whether the hitter will start. The confirmed board requires a complete posted lineup. Started, postponed and suspended games are excluded. The code reports missing entries instead of inventing a lineup. Historical lineups come from today's reconstruction of the MLB feed, not a saved pregame snapshot; the backtest assumes the listed starting lineup and pitcher were known just before first pitch. Late scratches require a refresh.

No current-game batting results, actual plate-appearance totals or postgame weather are used as predictors.

In [ ]:
def mlb_json(path, params=None, cache_file=None):
    """Read MLB's public JSON feed. Only completed historical data are cached."""
    if cache_file is not None and cache_file.exists():
        return json.loads(cache_file.read_text())
    url = "https://statsapi.mlb.com/api/" + path
    if params:
        url += "?" + urlencode(params)
    try:
        with urlopen(url, timeout=60) as response:
            result = json.load(response)
    except Exception as error:
        raise RuntimeError(f"Could not read MLB {path}: {error}") from error
    if cache_file is not None:
        cache_file.write_text(json.dumps(result))
    return result


def get_schedule(start, end, historical=True):
    """One request per month; lineups stay in their original batting order."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    games = []
    for month in pd.period_range(start, end, freq="M"):
        left = max(start, month.start_time).strftime("%Y-%m-%d")
        right = min(end, month.end_time.normalize()).strftime("%Y-%m-%d")
        cache_file = CACHE / f"schedule_{left}_{right}.json" if historical else None
        data = mlb_json("v1/schedule", {
            "sportId": 1, "startDate": left, "endDate": right,
            "gameTypes": "R", "hydrate": "probablePitcher,lineups,team"
        }, cache_file)
        for day in data.get("dates", []):
            games.extend(day.get("games", []))
    return games


def lineup_rows(games, historical=True):
    """Create one row per starting hitter/game; do not guess missing lineups."""
    rows, skipped = [], []
    for game in games:
        state = game.get("status", {}).get("abstractGameState")
        detail = game.get("status", {}).get("detailedState", "")
        if game.get("gameType") != "R":
            continue
        if historical and state != "Final":
            continue
        if not historical and state != "Preview":
            continue
        # A suspended game spans multiple dates, breaking a simple daily cutoff.
        if (game.get("resumeDate") or game.get("resumedFrom")
                or any(word in detail for word in ["Suspended", "Postponed", "Cancelled"])):
            skipped.append((game["gamePk"], "suspended/resumed game"))
            continue
        for side, other in [("home", "away"), ("away", "home")]:
            lineup = game.get("lineups", {}).get(side + "Players", [])
            starter = game["teams"][other].get("probablePitcher", {})
            if len(lineup) != 9 or not starter.get("id"):
                skipped.append((game["gamePk"], side + ": lineup/starter unavailable"))
                continue
            for order, person in enumerate(lineup, start=1):
                rows.append({
                    "game_pk": game["gamePk"], "game_date": game["officialDate"],
                    "game_number": game.get("gameNumber", 1),
                    "batter": person["id"], "player": person["fullName"],
                    "starter": starter["id"], "opposing_starter": starter["fullName"],
                    "team_id": game["teams"][side]["team"]["id"],
                    "opponent_id": game["teams"][other]["team"]["id"],
                    "team": game["teams"][side]["team"].get("abbreviation", side),
                    "venue": str(game["venue"]["id"]),
                    "venue_name": game["venue"]["name"],
                    "batting_order": order, "is_home": int(side == "home"),
                    "night_game": int(game.get("dayNight") == "night"),
                })
    result = pd.DataFrame(rows)
    if not result.empty:
        result["game_date"] = pd.to_datetime(result["game_date"])
        if result.duplicated(["game_pk", "batter"]).any():
            raise ValueError("Duplicate hitter/game in the lineup feed.")
    if skipped:
        print(f"Skipped {len(skipped)} game/team entries; first examples: {skipped[:4]}")
    return result


def get_people(ids):
    """Use stable MLB IDs to avoid matching the wrong player by name."""
    file = CACHE / "player_handedness.json"
    saved = json.loads(file.read_text()) if file.exists() else {}
    missing = sorted({int(x) for x in ids if str(int(x)) not in saved})
    for offset in range(0, len(missing), 150):
        data = mlb_json("v1/people", {"personIds": ",".join(map(str, missing[offset:offset+150]))})
        for person in data.get("people", []):
            saved[str(person["id"])] = {
                "bat": person.get("batSide", {}).get("code", "U"),
                "throw": person.get("pitchHand", {}).get("code", "U")}
    file.write_text(json.dumps(saved))
    return saved


def add_handedness(rows, people):
    rows = rows.copy()
    rows["pitcher_hand"] = rows.starter.map(lambda x: people.get(str(x), {}).get("throw", "U"))
    rows["stand"] = rows.batter.map(lambda x: people.get(str(x), {}).get("bat", "U"))
    switch = rows.stand.eq("S")
    rows.loc[switch, "stand"] = rows.loc[switch, "pitcher_hand"].map({"R": "L", "L": "R"}).fillna("U")
    rows["platoon"] = rows.stand + "_vs_" + rows.pitcher_hand
    rows["park_side"] = rows.venue + "_" + rows.stand
    rows["month"] = rows.game_date.dt.month
    return rows

## 5. Download pitch data and identify completed plate appearances

Statcast data are downloaded in seven-day blocks and compressed summaries are cached. All identified pitches count toward a pitcher's pitch mix. The outcome table keeps only one terminal result for each completed plate appearance; a pickoff is not another plate appearance.

The target later becomes `1` if the starting hitter hit at least one HR in that game and `0` otherwise. Two home runs still count as `1` for this question.

If a download fails, the code stops and names the date block. It never replaces real records with simulated data. If access fails repeatedly, you will need an accessible Statcast export or a different environment before training can finish.

In [ ]:
RAW_COLUMNS = ["game_date", "game_pk", "game_type", "batter", "pitcher",
               "at_bat_number", "pitch_number", "events", "pitch_type",
               "stand", "p_throws", "inning_topbot", "launch_speed",
               "launch_angle", "launch_speed_angle", "bb_type",
               "estimated_woba_using_speedangle", "release_speed"]
COUNT_COLUMNS = ["pa", "hr", "k", "bb", "bbe", "ev_n", "ev_sum", "hard",
                 "barrel_n", "barrel", "fly", "xw_n", "xw_sum"]


def clean_pitches(raw):
    required = set(RAW_COLUMNS) - {"estimated_woba_using_speedangle", "launch_angle"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Statcast columns missing: {sorted(missing)}")
    p = raw.reindex(columns=RAW_COLUMNS).copy()
    p = p[p.game_type.eq("R")].copy()
    p["game_date"] = pd.to_datetime(p.game_date).dt.normalize()
    for col in ["game_pk", "batter", "pitcher", "at_bat_number", "pitch_number"]:
        p[col] = pd.to_numeric(p[col], errors="raise").astype("int64")
    for col in ["launch_speed", "launch_speed_angle", "release_speed",
                "estimated_woba_using_speedangle"]:
        # Convert nullable pandas numbers to ordinary NaN-based floats as well.
        p[col] = pd.to_numeric(p[col], errors="coerce").astype("float64")
    p = p.drop_duplicates(["game_pk", "at_bat_number", "pitch_number"])
    group = p.pitch_type.replace({"KC": "CU", "CS": "CU", "FA": "FF"})
    p["pitch_group"] = group.where(group.isin(PITCHES), "OTHER")
    p["stand"] = p.stand.fillna("U")
    p["p_throws"] = p.p_throws.fillna("U")
    return p


def summarize_chunk(p):
    # Keep one result per completed PA. Pickoffs and stolen bases are not PAs.
    pa = p[p.events.isin(PA_EVENTS)].sort_values(["game_pk", "at_bat_number", "pitch_number"])
    pa = pa.drop_duplicates(["game_pk", "at_bat_number"], keep="last").copy()
    pa["pa"] = 1
    pa["hr"] = pa.events.eq("home_run").astype(int)
    pa["k"] = pa.events.isin(["strikeout", "strikeout_double_play"]).astype(int)
    pa["bb"] = pa.events.isin(["walk", "intent_walk", "intentional_walk"]).astype(int)
    pa["bbe"] = pa.bb_type.notna().astype(int)
    pa["ev_n"] = pa.launch_speed.notna().astype(int)
    pa["ev_sum"] = pa.launch_speed.fillna(0)
    pa["hard"] = pa.launch_speed.ge(95).astype(int)
    pa["barrel_n"] = pa.launch_speed_angle.notna().astype(int)
    pa["barrel"] = pa.launch_speed_angle.eq(6).astype(int)
    pa["fly"] = pa.bb_type.eq("fly_ball").fillna(False).astype(int)
    pa["xw_n"] = pa.estimated_woba_using_speedangle.notna().astype(int)
    pa["xw_sum"] = pa.estimated_woba_using_speedangle.fillna(0)
    pa = pa[["game_date", "game_pk", "batter", "pitcher", "at_bat_number",
             "stand", "p_throws", "inning_topbot", "pitch_group"] + COUNT_COLUMNS]
    # Pitch usage includes all identified pitches, not only balls put in play.
    pitches = p[p.pitch_type.notna()].copy()
    pitches["n"] = 1
    pitches["v_n"] = pitches.release_speed.notna().astype(int)
    pitches["v_sum"] = pitches.release_speed.fillna(0)
    pitch_daily = pitches.groupby(["game_date", "pitcher", "stand", "pitch_group"],
                                  as_index=False)[["n", "v_n", "v_sum"]].sum()
    return pa, pitch_daily


def get_statcast(start, end, games):
    """Download seven-day blocks with retries and a daily fallback."""
    import time

    try:
        from pybaseball import statcast
    except ImportError as error:
        raise RuntimeError(
            "Run the installation cell first: %pip install pybaseball"
        ) from error

    def request_with_retries(start_date, end_date, label, attempts=3):
        """Retry a Statcast request before declaring it failed."""
        last_error = None

        for attempt in range(1, attempts + 1):
            try:
                raw = statcast(
                    start_date,
                    end_date,
                    verbose=False,
                    parallel=False
                )

                if raw is None or raw.empty:
                    raise ValueError("No pitches were returned.")

                return raw

            except Exception as error:
                last_error = error

                if attempt < attempts:
                    wait_seconds = 5 * attempt
                    print(
                        f"{label} failed on attempt {attempt}. "
                        f"Retrying in {wait_seconds} seconds..."
                    )
                    time.sleep(wait_seconds)

        raise last_error

    all_pa = []
    all_pitch = []

    available = {
        game["officialDate"]
        for game in games
        if game.get("gameType") == "R"
        and game.get("status", {}).get("abstractGameState") == "Final"
    }

    end = pd.Timestamp(end)

    for left in pd.date_range(start, end, freq="7D"):
        right = min(left + pd.Timedelta(days=6), end)

        dates_in_block = [
            day
            for day in available
            if left.strftime("%Y-%m-%d")
            <= day
            <= right.strftime("%Y-%m-%d")
        ]

        if not dates_in_block:
            continue

        tag = f"{left:%Y-%m-%d}_{right:%Y-%m-%d}"

        pa_file = CACHE / f"pa_{tag}.csv.gz"
        pitch_file = CACHE / f"pitches_{tag}.csv.gz"

        if pa_file.exists() and pitch_file.exists():
            pa = pd.read_csv(pa_file, parse_dates=["game_date"])
            pitch_daily = pd.read_csv(
                pitch_file,
                parse_dates=["game_date"]
            )

        else:
            print("Downloading", tag)

            try:
                # First try downloading the whole week.
                raw = request_with_retries(
                    str(left.date()),
                    str(right.date()),
                    tag
                )

            except Exception as weekly_error:
                print(
                    f"Weekly request for {tag} failed. "
                    "Trying one day at a time..."
                )

                daily_results = []

                for day in pd.date_range(left, right, freq="D"):
                    day_text = day.strftime("%Y-%m-%d")

                    if day_text not in available:
                        continue

                    try:
                        daily_raw = request_with_retries(
                            day_text,
                            day_text,
                            day_text
                        )
                        daily_results.append(daily_raw)
                        time.sleep(1)

                    except Exception as daily_error:
                        raise RuntimeError(
                            f"Statcast also failed for {day_text}. "
                            f"Weekly error: {weekly_error}. "
                            f"Daily error: {daily_error}"
                        ) from daily_error

                if not daily_results:
                    raise RuntimeError(
                        f"No Statcast data could be downloaded for {tag}."
                    )

                raw = pd.concat(daily_results, ignore_index=True)

            pa, pitch_daily = summarize_chunk(clean_pitches(raw))

            pa.to_csv(pa_file, index=False)
            pitch_daily.to_csv(pitch_file, index=False)

        all_pa.append(pa)
        all_pitch.append(pitch_daily)

    if not all_pa:
        raise ValueError(
            "No completed regular-season Statcast games in this date range."
        )

    return (
        pd.concat(all_pa, ignore_index=True)
        .drop_duplicates(["game_pk", "at_bat_number"]),

        pd.concat(all_pitch, ignore_index=True)
    )

## 6. Join the records and create the outcome

The starter is checked against the first recorded opposing pitcher. Later pitchers count as relief appearances. A listed hitter with no recorded PA is excluded and reported for review; missing records are not automatically labeled as a zero-HR game.

The code checks for whole games missing from the Statcast download. This is a useful first data-quality check, but a production version should also reconcile every game's PA and HR totals to official box scores to catch partial downloads.

In [ ]:
def connect_history(pa, pitch_daily, rows):
    """Verify suspect historical starter listings using official box scores."""
    rows = rows.copy()
    plays = pa.copy()
    plays["is_home"] = plays.inning_topbot.eq("Bot").astype(int)
    missing = set(rows.game_pk) - set(plays.game_pk)
    if missing:
        raise ValueError(f"Missing Statcast data for {len(missing)} scheduled games, e.g. {sorted(missing)[:5]}.")

    # The pitcher credited with the first completed PA may differ from the
    # announced starter, or even from the real starter after a mid-PA change.
    first = plays.sort_values(["game_pk", "at_bat_number"]).drop_duplicates(
        ["game_pk", "is_home"])[["game_pk", "is_home", "pitcher"]]
    listed = rows[["game_pk", "is_home", "starter"]].drop_duplicates()
    check = listed.merge(first, on=["game_pk", "is_home"], how="left", validate="one_to_one")
    suspect_games = check.loc[~check.pitcher.eq(check.starter), "game_pk"].unique()

    excluded, audit, corrected = set(), [], 0
    for game_pk in suspect_games:
        try:
            box = mlb_json(f"v1/game/{int(game_pk)}/boxscore",
                           cache_file=CACHE / f"boxscore_{int(game_pk)}.json")
            updates = []
            # Check BOTH teams before accepting any changes to this game.
            for batting_side, is_home, fielding_side in [("away", 0, "home"), ("home", 1, "away")]:
                team_plays = plays[(plays.game_pk == game_pk) & (plays.is_home == is_home)]
                official = box["teams"][batting_side]["teamStats"]["batting"]
                if (len(team_plays) != int(official["plateAppearances"])
                        or int(team_plays.hr.sum()) != int(official["homeRuns"])):
                    raise ValueError(f"{batting_side} PA/HR totals differ from the official box score")

                # gamesStarted identifies the actual starter, including openers.
                players = box["teams"][fielding_side]["players"].values()
                starters = [p for p in players
                            if int(p.get("stats", {}).get("pitching", {}).get("gamesStarted", 0)) == 1]
                if len(starters) != 1:
                    raise ValueError(f"Cannot identify one official {fielding_side} starter")
                person = starters[0]["person"]
                updates.append((is_home, int(person["id"]), person["fullName"]))
        except (RuntimeError, KeyError, TypeError, ValueError) as error:
            excluded.add(game_pk)
            audit.append({"game_pk": int(game_pk), "status": "excluded", "reason": str(error)})
            continue

        changes = []
        for is_home, pitcher_id, pitcher_name in updates:
            mask = (rows.game_pk == game_pk) & (rows.is_home == is_home)
            if mask.any() and rows.loc[mask, "starter"].ne(pitcher_id).any():
                old_ids = rows.loc[mask, "starter"].unique().tolist()
                changes.append(f"batting is_home={is_home}: {old_ids} -> {pitcher_id}")
                corrected += 1
            rows.loc[mask, "starter"] = pitcher_id
            rows.loc[mask, "opposing_starter"] = pitcher_name
        audit.append({"game_pk": int(game_pk), "status": "verified",
                      "reason": "; ".join(changes) or "Official starter confirmed; opening PA pitcher differs"})

    if len(suspect_games):
        rows = rows[~rows.game_pk.isin(excluded)].copy()
        audit_file = CACHE / "starter_reconciliation.csv"
        pd.DataFrame(audit).to_csv(audit_file, index=False)
        print(f"Checked {len(suspect_games)} historical games: corrected {corrected} starter listings; "
              f"excluded {len(excluded)} games from training and PA history.")
        print("Details:", audit_file)
        if rows.empty:
            raise ValueError("No verified historical games remain. Review starter_reconciliation.csv.")
        # A different starter may have a different throwing hand. Refresh ALL
        # related features, including the batting side of a switch hitter.
        people = get_people(set(rows.batter) | set(rows.starter))
        rows = add_handedness(rows, people)

    # Rebuild the join so both training rows and bullpen labels use the verified
    # starter ID. Merely removing the old error would leave incorrect matchups.
    context = rows[["game_pk", "is_home", "team_id", "opponent_id", "starter"]].drop_duplicates()
    result = plays.merge(context, on=["game_pk", "is_home"], how="inner", validate="many_to_one")
    result["relief"] = result.pitcher.ne(result.starter)
    outcomes = result.groupby(["game_pk", "batter"], as_index=False).agg(
        hr_count=("hr", "sum"), actual_pa=("pa", "sum"))
    labeled = rows.merge(outcomes, on=["game_pk", "batter"], how="left", validate="one_to_one")
    no_pa = labeled.actual_pa.isna()
    if no_pa.any():
        print(f"Excluded {no_pa.sum()} lineup entries without a recorded PA; inspect scratches/data coverage.")
    labeled = labeled[~no_pa].reset_index(drop=True)
    labeled["hit_hr"] = labeled.hr_count.gt(0).astype(int)
    # The existing pitch cache is already aggregated by day, not by game.
    # Keep those recorded prior-day pitches for repertoire estimates; excluded
    # games contribute no labels or PA/contact statistics to the model.
    return result.reset_index(drop=True), pitch_daily, labeled


## 7. Build history without looking into the future

This is the main bookkeeping section. `History.before()` adds up information from **before** each prediction date. The rolling window is `[date − window, date)`, so nothing from the target day is included. Both games of a doubleheader use the same prior-day history.

`smooth()` pulls a small sample toward a broader prior. For example, a hitter with only a few measured contacts against sliders does not get treated as an established slider specialist. The priors are built from earlier data. The starting choices of 100 PA, 50 measured contacts, or 40 pitch-specific contacts are smoothing strengths, **not learned model weights**; tune them later on validation dates only.

No exact batter-versus-pitcher history is used in this version. Pitch-type and handedness history provide broader samples.

In [ ]:
class History:
    """Daily cumulative sums let us ask: what was known BEFORE this date?"""
    def __init__(self, frame, keys, values):
        self.keys, self.values, self.series = keys, values, {}
        d = frame.groupby(keys + ["game_date"], as_index=False, dropna=False)[values].sum()
        for key, part in d.groupby(keys, sort=False, dropna=False):
            key = key if isinstance(key, tuple) else (key,)
            part = part.sort_values("game_date")
            dates = part.game_date.to_numpy(dtype="datetime64[D]")
            cumulative = np.vstack([np.zeros(len(values)), np.cumsum(part[values].to_numpy(float), axis=0)])
            self.series[key] = (dates, cumulative)

    def before(self, query, days):
        out = np.zeros((len(query), len(self.values)))
        # groupby.indices gives row POSITIONS, so non-consecutive indices work.
        for key, positions in query.groupby(self.keys, sort=False, dropna=False).indices.items():
            key = key if isinstance(key, tuple) else (key,)
            if key not in self.series:
                continue
            dates, cumulative = self.series[key]
            targets = query.iloc[positions].game_date.to_numpy(dtype="datetime64[D]")
            # side='left' excludes EVERY game on the target date, even game 1
            # of a doubleheader. The window is [date - days, date).
            stop = np.searchsorted(dates, targets, side="left")
            start = np.searchsorted(dates, targets - np.timedelta64(days, "D"), side="left")
            out[positions] = cumulative[stop] - cumulative[start]
        return pd.DataFrame(out, index=query.index, columns=self.values)


def smooth(numerator, denominator, prior, strength):
    """A small sample stays close to its broader prior; a large one speaks more."""
    return (numerator + strength * prior) / (denominator + strength)


def pitch_wide(frame, keys, values):
    wide = frame.pivot_table(index=keys + ["game_date"], columns="pitch_group",
                            values=values, aggfunc="sum", fill_value=0)
    wide.columns = [f"{metric}_{pitch}" for metric, pitch in wide.columns]
    wanted = [f"{metric}_{pitch}" for metric in values for pitch in PITCHES]
    return wide.reindex(columns=wanted, fill_value=0).reset_index(), wanted


def build_stores(pa, pitches, labeled):
    pa = pa.assign(league=1)
    stores = {
        "league": History(pa, ["league"], COUNT_COLUMNS),
        "hitter": History(pa, ["batter"], COUNT_COLUMNS),
        "hitter_split": History(pa.rename(columns={"p_throws": "pitcher_hand"}), ["batter", "pitcher_hand"], COUNT_COLUMNS),
        "pitcher": History(pa.assign(starter=pa.pitcher), ["starter"], COUNT_COLUMNS),
        "pitcher_split": History(pa.assign(starter=pa.pitcher), ["starter", "stand"], COUNT_COLUMNS),
        "bullpen": History(pa[pa.relief], ["opponent_id"], COUNT_COLUMNS),
    }
    game_history = labeled.assign(games=1)[["game_date", "batter", "games", "hit_hr"]]
    stores["game_hitter"] = History(game_history, ["batter"], ["games", "hit_hr"])
    stores["game_league"] = History(game_history.assign(league=1), ["league"], ["games", "hit_hr"])
    wide, values = pitch_wide(pitches.rename(columns={"pitcher": "starter"}), ["starter", "stand"], ["n", "v_n", "v_sum"])
    stores["mix_split"] = History(wide, ["starter", "stand"], values)
    stores["mix_all"] = History(wide, ["starter"], values)
    stores["mix_league"] = History(wide.assign(league=1), ["league", "stand"], values)
    wide, values = pitch_wide(pa.rename(columns={"p_throws": "pitcher_hand"}), ["batter", "pitcher_hand"], ["barrel", "barrel_n", "hard", "ev_n"])
    stores["contact_split"] = History(wide, ["batter", "pitcher_hand"], values)
    return stores

## 8. Combine hitter, pitcher and pitch-type information

Pitch usage starts with the pitcher's prior 365-day repertoire, including left/right splits. The most recent 60 days are then blended toward that longer record. Hitter performance against each pitch uses prior 365-day measured contact, split by pitcher handedness.

The pitch-fit feature is a usage-weighted summary. As a **hypothetical illustration**, if a pitcher throws 60% fastballs and 40% sliders, and the hitter's smoothed barrel rates against those pitches are 10% and 5%, the feature is `0.60 × 0.10 + 0.40 × 0.05 = 0.08`.

**That 8% is not an 8% HR probability.** Pitch usage and contact-quality rates have different denominators, and a hitter might swing through or take a pitch. This is an input the regression can learn from, alongside strikeout rate, HR rate and the other features. A full pitch-sequence/contact model would be a later upgrade.

All 14-, 30-, 60- and 365-day windows are calendar-day windows, not the last N games. The sample counts stay in the model so it can distinguish a well-supported rate from a thin sample.

In [ ]:
def features(rows, stores):
    q = rows.copy().reset_index(drop=True).assign(league=1)
    f = q.copy()
    lg = stores["league"].before(q, 365)
    # Numerical fallbacks only cover an empty warmup period. Training starts
    # after a full warmup year, so league rates normally come from past data.
    defaults = {"hr_rate": .03, "k_rate": .22, "bb_rate": .08,
                "barrel_rate": .08, "hard_rate": .40, "ev": 88.,
                "fly_rate": .25, "xwobacon": .37}
    metrics = {"hr_rate": ("hr", "pa", 100), "k_rate": ("k", "pa", 100),
               "bb_rate": ("bb", "pa", 100), "barrel_rate": ("barrel", "barrel_n", 50),
               "hard_rate": ("hard", "ev_n", 50), "ev": ("ev_sum", "ev_n", 50),
               "fly_rate": ("fly", "bbe", 50), "xwobacon": ("xw_sum", "xw_n", 50)}
    league_rates = {name: (lg[num] / lg[den].replace(0, np.nan)).fillna(defaults[name])
                    for name, (num, den, _) in metrics.items()}
    long_hitter = stores["hitter"].before(q, 365)
    f["hitter_pa_365"] = long_hitter.pa
    for days in [365, 30, 14]:
        totals = long_hitter if days == 365 else stores["hitter"].before(q, days)
        for name, (num, den, strength) in metrics.items():
            prior = league_rates[name] if days == 365 else f[f"h_{name}_365"]
            f[f"h_{name}_{days}"] = smooth(totals[num], totals[den], prior, strength)
        f[f"hitter_pa_{days}"] = totals.pa
    for name in ["barrel_rate", "ev", "hard_rate"]:
        f[f"recent_{name}_change"] = f[f"h_{name}_30"] - f[f"h_{name}_365"]
    split = stores["hitter_split"].before(q, 365)
    f["hitter_split_pa"] = split.pa
    for name in ["hr_rate", "barrel_rate", "k_rate"]:
        num, den, strength = metrics[name]
        f["h_split_" + name] = smooth(split[num], split[den], f[f"h_{name}_365"], strength)
    for store_name, prefix, days in [("pitcher", "p", 365), ("pitcher", "p_recent", 60),
                                     ("pitcher_split", "p_split", 365), ("bullpen", "bullpen", 60)]:
        totals = stores[store_name].before(q, days)
        f[prefix + "_pa"] = totals.pa
        for name in ["hr_rate", "barrel_rate", "hard_rate", "k_rate", "fly_rate"]:
            num, den, strength = metrics[name]
            prior = f["p_" + name] if prefix in ["p_recent", "p_split"] else league_rates[name]
            f[prefix + "_" + name] = smooth(totals[num], totals[den], prior, strength)
    game_lg = stores["game_league"].before(q, 365)
    game_h = stores["game_hitter"].before(q, 365)
    f["league_game_hr_rate"] = (game_lg.hit_hr / game_lg.games.replace(0, np.nan)).fillna(.11)
    f["h_game_hr_rate"] = smooth(game_h.hit_hr, game_h.games, f.league_game_hr_rate, 30)
    mix_lg = stores["mix_league"].before(q, 365)
    mix_all = stores["mix_all"].before(q, 365)
    mix_split_long = stores["mix_split"].before(q, 365)
    mix_split = stores["mix_split"].before(q, 60)
    contact = stores["contact_split"].before(q, 365)
    ncols = ["n_" + p for p in PITCHES]
    lg_total, all_total, split_long_total, split_total = [x[ncols].sum(axis=1) for x in [mix_lg, mix_all, mix_split_long, mix_split]]
    f["pitcher_split_pitches"] = split_total
    f["pitch_fit_barrel"] = 0.
    f["pitch_fit_hard"] = 0.
    for pitch in PITCHES:
        lg_share = (mix_lg["n_"+pitch] / lg_total.replace(0, np.nan)).fillna(1 / len(PITCHES))
        all_share = smooth(mix_all["n_"+pitch], all_total, lg_share, 200)
        split_long_share = smooth(mix_split_long["n_"+pitch], split_long_total, all_share, 150)
        share = smooth(mix_split["n_"+pitch], split_total, split_long_share, 150)
        f["mix_" + pitch] = share
        barrel = smooth(contact["barrel_"+pitch], contact["barrel_n_"+pitch], f.h_split_barrel_rate, 40)
        hard = smooth(contact["hard_"+pitch], contact["ev_n_"+pitch], f.h_hard_rate_365, 40)
        f["contact_barrel_" + pitch] = barrel
        f["pitch_fit_barrel"] += share * barrel
        f["pitch_fit_hard"] += share * hard
    # These are matchup FEATURES, not probabilities of a home run. Pitch shares
    # and conditional contact rates have different denominators.
    f["pitch_fit_barrel_edge"] = f.pitch_fit_barrel - f.h_split_barrel_rate
    v_num = mix_split[["v_sum_"+p for p in PITCHES]].sum(axis=1)
    v_den = mix_split[["v_n_"+p for p in PITCHES]].sum(axis=1)
    f["pitcher_avg_velocity"] = v_num / v_den.replace(0, np.nan)
    f["low_hitter_history"] = (long_hitter.pa < 100).astype(int)
    f["low_pitcher_history"] = (f.p_pa < 100).astype(int)
    return f

## 9. Fit an explainable model and test it on later games

The target is binary, so the first model is **regularized logistic regression**. It estimates the chance of a game with at least one HR. It learns the statistical weights from data; there is no manually assigned '30% matchup / 20% recent form' score.

Two feature sets are compared: a simple hitter-power/batting-order baseline and the full model. Each gets three regularization settings. The best validation log loss wins. If the simple version wins, the notebook says so. Adding more baseball detail does not guarantee better predictions.

| Period | Role |
|---|---|
| 2022 | Warmup history only |
| 2023–2024 | Fit coefficients and preprocessing |
| Jan.–June 2025 | Select the feature set and regularization strength |
| July–August 2025 | Fit the probability calibration step |
| September–December 2025, regular-season games only | Final held-out evaluation |

Earlier completed games within a validation/test period can enter later rolling features, just as they would in daily use. The coefficients and calibrator remain fixed during the held-out test. Games are never randomly shuffled across time.

Use **log loss** and **Brier score** to evaluate probability predictions; lower is better. Average precision and the actual HR rate among each day's top 10 ranked hitter/game entries describe ranking performance. Check the calibration table/plot as well: among predictions around 20%, did roughly 20% homer? A high ordinary classification accuracy can come from predicting 'no HR' for almost everyone.

Calibration and evaluation are explained in [scikit-learn's calibration guide](https://scikit-learn.org/stable/modules/calibration.html). Neither a good calibration plot nor a lower Brier score alone establishes that the model ranks hitters better.

Once the design is selected, the daily model refits on more recent history, reserving the latest 60 days for calibration. The 2025 test describes the earlier frozen model, not an independent test of that later refit.

In [ ]:
BASE_NUMERIC = ["h_game_hr_rate", "h_hr_rate_365", "batting_order", "is_home"]
CATEGORIES = ["platoon", "park_side"]


def numeric_columns(frame, family):
    if family == "baseline":
        return BASE_NUMERIC
    prefixes = ("h_", "p_", "p_recent_", "p_split_", "bullpen_", "hitter_",
                "recent_", "pitch_fit_", "mix_", "low_")
    extra = [c for c in frame.columns if c.startswith(prefixes)]
    return sorted(set(extra + BASE_NUMERIC + ["night_game", "month", "pitcher_avg_velocity", "pitcher_split_pitches"]))


def make_model(frame, family, c):
    numeric = numeric_columns(frame, family)
    parts = [("numeric", make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True), StandardScaler()), numeric)]
    if family == "full":
        parts.append(("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORIES))
    prep = ColumnTransformer(parts)
    # Do not balance the classes: we want real probabilities of a rare event.
    return make_pipeline(prep, LogisticRegression(C=c, max_iter=1500, solver="lbfgs"))


def fit_and_test(frame):
    train = frame[(frame.game_date >= TRAIN_START) & (frame.game_date < VALID_START)]
    valid = frame[(frame.game_date >= VALID_START) & (frame.game_date < CALIBRATION_START)]
    cal = frame[(frame.game_date >= CALIBRATION_START) & (frame.game_date < TEST_START)]
    test = frame[(frame.game_date >= TEST_START) & (frame.game_date <= TEST_END)]
    for name, part in [("training", train), ("validation", valid), ("calibration", cal), ("test", test)]:
        if len(part) < 1000 or part.hit_hr.nunique() != 2:
            raise ValueError(f"Insufficient {name} data: {len(part)} rows. Check dates and download coverage.")
    trials, candidates = [], {}
    for family in ["baseline", "full"]:
        for c in [.1, 1., 10.]:
            model = make_model(train, family, c).fit(train, train.hit_hr)
            score = log_loss(valid.hit_hr, model.predict_proba(valid)[:, 1], labels=[0, 1])
            trials.append({"family": family, "C": c, "validation_log_loss": score})
            candidates[(family, c)] = model
    trials = pd.DataFrame(trials).sort_values("validation_log_loss").reset_index(drop=True)
    chosen = trials.iloc[0]
    model = candidates[(chosen.family, chosen.C)]
    # Calibration sees a different period from both training and model selection.
    calibrator = LogisticRegression(C=1000, max_iter=1000).fit(
        model.decision_function(cal).reshape(-1, 1), cal.hit_hr)
    if calibrator.coef_[0, 0] <= 0:
        raise ValueError("Calibration reversed the ranking. Investigate data/model before using predictions.")
    scored = test.copy()
    scored["prediction"] = calibrator.predict_proba(model.decision_function(test).reshape(-1, 1))[:, 1]
    scores = []
    for label, prediction in [("Selected model", scored.prediction),
                               ("Past hitter HR/game", scored.h_game_hr_rate),
                               ("Past league HR/game", scored.league_game_hr_rate)]:
        ranked = scored.assign(score=prediction).sort_values(["game_date", "score"], ascending=[True, False])
        top = ranked.groupby("game_date").head(10)
        scores.append({"model": label, "log_loss": log_loss(scored.hit_hr, prediction, labels=[0, 1]),
                       "brier_score": brier_score_loss(scored.hit_hr, prediction),
                       "average_precision": average_precision_score(scored.hit_hr, prediction),
                       "top10_hr_rate": top.hit_hr.mean(), "test_rows": len(scored)})
    bins = pd.qcut(scored.prediction, q=10, duplicates="drop")
    calibration = scored.groupby(bins, observed=True).agg(
        predicted=("prediction", "mean"), actual=("hit_hr", "mean"), n=("hit_hr", "size"))
    package = {"model": model, "calibrator": calibrator, "family": chosen.family, "C": float(chosen.C)}
    return package, trials, pd.DataFrame(scores), calibration, scored


def fit_daily_model(frame, choice, target_date):
    """Refit the chosen specification; reserve the latest 60 days to calibrate."""
    cutoff = pd.Timestamp(target_date) - pd.Timedelta(days=60)
    train = frame[(frame.game_date >= TRAIN_START) & (frame.game_date < cutoff)]
    cal = frame[(frame.game_date >= cutoff) & (frame.game_date < target_date)]
    if len(cal) < 1000 or cal.hit_hr.nunique() < 2:
        raise ValueError("Not enough recent calibration games. For early-season use, extend the calibration window.")
    model = make_model(train, choice["family"], choice["C"]).fit(train, train.hit_hr)
    calibrator = LogisticRegression(C=1000, max_iter=1000).fit(model.decision_function(cal).reshape(-1, 1), cal.hit_hr)
    if calibrator.coef_[0, 0] <= 0:
        raise ValueError("Non-positive calibration slope: investigate before ranking players.")
    return {**choice, "model": model, "calibrator": calibrator,
            "trained_before": cutoff.strftime("%Y-%m-%d"), "as_of": str(target_date)}


def predict_rankings(frame, fitted):
    result = frame.copy()
    log_odds = fitted["model"].decision_function(frame).reshape(-1, 1)
    result["hr_probability"] = fitted["calibrator"].predict_proba(log_odds)[:, 1]
    result["hr_chance_pct"] = 100 * result.hr_probability
    result["hitter_barrel_pct"] = 100 * result.h_barrel_rate_365
    result["recent_barrel_pct"] = 100 * result.h_barrel_rate_30
    result["pitch_fit_edge_pp"] = 100 * result.pitch_fit_barrel_edge
    result["starter_hr_per_100_pa"] = 100 * result.p_split_hr_rate
    result["most_common_pitch"] = result[["mix_"+p for p in PITCHES]].idxmax(axis=1).str.replace("mix_", "", regex=False).map(PITCH_NAMES)
    result["history_flag"] = np.where(result.low_hitter_history | result.low_pitcher_history, "Limited history", "")
    result["model_family"] = fitted["family"]
    result = result.sort_values("hr_probability", ascending=False).reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result)+1))
    return result


def player_day_rankings(game_rankings):
    """Combine the covered games; a doubleheader uses an independence approximation."""
    days = []
    for (date, batter), part in game_rankings.groupby(["game_date", "batter"]):
        days.append({"date": date.strftime("%Y-%m-%d"), "batter": batter,
                     "player": part.player.iloc[0], "team": part.team.iloc[0],
                     "games_covered": len(part),
                     "hr_chance_pct": 100 * (1 - np.prod(1 - part.hr_probability)),
                     "opposing_starters": "; ".join(part.opposing_starter),
                     "note": "Across-game independence approximation" if len(part) > 1 else "One posted upcoming game"})
    result = pd.DataFrame(days).sort_values("hr_chance_pct", ascending=False).reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    return result

## 10. Define the full workflow

The next cell connects the functions. It only defines the workflow; the following cell actually runs it. The early function produces an active-roster file conditional on each hitter starting. The confirmed-lineup function produces one file per hitter/game and a second file per player/day.

A player in two posted upcoming games gets an approximate daily chance of `1 − (1 − p_game1)(1 − p_game2)`. This assumes independent game outcomes conditional on the estimates and is labeled as an approximation. Unposted second-game lineups and games already underway are not included. The game-level probabilities should be evaluated before relying on the doubleheader combination.

In [ ]:
# This block is a notebook cell. It is intentionally not run when this file is imported.
def run_history(target_date=TARGET_DATE):
    if pd.Timestamp(target_date) <= pd.Timestamp(TEST_END):
        raise ValueError("This preset needs a 2026-or-later prediction date. Change all time splits together for an earlier study.")
    history_end = (pd.Timestamp(target_date) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    schedule = get_schedule(HISTORY_START, history_end)
    historical_rows = lineup_rows(schedule)
    if historical_rows.empty:
        raise ValueError("No complete historical starting lineups returned.")
    people = get_people(set(historical_rows.batter) | set(historical_rows.starter))
    historical_rows = add_handedness(historical_rows, people)
    pa, pitches = get_statcast(HISTORY_START, history_end, schedule)
    pa, pitches, labeled = connect_history(pa, pitches, historical_rows)
    stores = build_stores(pa, pitches, labeled)
    training_rows = labeled[labeled.game_date >= TRAIN_START].reset_index(drop=True)
    data = features(training_rows, stores)
    package, trials, metrics, calibration, test = fit_and_test(data)
    package["history_end"] = history_end
    print(f"Historical sample: {len(data):,} starting hitter/games; {data.hit_hr.mean():.1%} with a home run.")
    print("Selected specification:", package["family"], "C =", package["C"])
    if package["family"] == "baseline":
        print("The matchup features did not beat the simpler model on validation data.")
    return stores, data, package, trials, metrics, calibration, test


def run_daily(stores, data, choice, target_date=TARGET_DATE):
    expected_end = (pd.Timestamp(target_date) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    if choice.get("history_end") != expected_end:
        raise ValueError("The prediction date changed or history is stale. Rerun run_history for this target date first.")
    schedule = get_schedule(target_date, target_date, historical=False)
    rows = lineup_rows(schedule, historical=False)
    if rows.empty:
        print("No upcoming games with a complete posted lineup and opposing starter yet. Try again closer to first pitch.")
        return pd.DataFrame()
    people = get_people(set(rows.batter) | set(rows.starter))
    rows = add_handedness(rows, people)
    future = features(rows, stores)
    model = fit_daily_model(data, choice, target_date)
    result = predict_rankings(future, model)
    result["generated_at_utc"] = datetime.now(ZoneInfo("UTC")).isoformat()
    result.to_csv(f"HR_Rankings_{target_date}.csv", index=False)
    player_day_rankings(result).to_csv(f"HR_Player_Day_Rankings_{target_date}.csv", index=False)
    print("Rankings are per game, conditional on the posted lineup. Refresh for scratches.")
    print("Only not-yet-started regular-season games are included.")
    return result


def active_roster_rows(games, target_date):
    """Create one hypothetical starter row for every active non-pitcher.

    These rows support an early-day P(HR | starts) ranking. They do not claim
    that every listed hitter will actually be in the starting lineup.
    """
    target_text = pd.Timestamp(target_date).strftime("%Y-%m-%d")
    rows, skipped = [], []
    roster_cache = {}

    for game in games:
        state = game.get("status", {}).get("abstractGameState")
        detail = game.get("status", {}).get("detailedState", "")
        if game.get("gameType") != "R" or state != "Preview":
            continue
        if (game.get("resumeDate") or game.get("resumedFrom")
                or any(word in detail for word in ["Suspended", "Postponed", "Cancelled"])):
            skipped.append((game.get("gamePk"), "suspended/postponed game"))
            continue

        for side, other in [("home", "away"), ("away", "home")]:
            starter = game["teams"][other].get("probablePitcher", {})
            if not starter.get("id"):
                skipped.append((game["gamePk"], side + ": opposing starter unavailable"))
                continue

            team = game["teams"][side]["team"]
            team_id = int(team["id"])
            if team_id not in roster_cache:
                roster_data = mlb_json(
                    f"v1/teams/{team_id}/roster",
                    {"rosterType": "active", "date": target_text, "hydrate": "person"},
                )
                roster_cache[team_id] = roster_data.get("roster", [])

            eligible = 0
            for entry in roster_cache[team_id]:
                person = entry.get("person", {})
                position = entry.get("position", {})
                # Position code 1 is pitcher. Two-way players remain eligible.
                if position.get("type") == "Pitcher" or str(position.get("code")) == "1":
                    continue
                if not person.get("id"):
                    continue
                eligible += 1
                rows.append({
                    "game_pk": game["gamePk"],
                    "game_date": game["officialDate"],
                    "game_number": game.get("gameNumber", 1),
                    "batter": int(person["id"]),
                    "player": person.get("fullName", str(person["id"])),
                    "starter": int(starter["id"]),
                    "opposing_starter": starter.get("fullName", str(starter["id"])),
                    "team_id": team_id,
                    "opponent_id": int(game["teams"][other]["team"]["id"]),
                    "team": team.get("abbreviation", side),
                    "venue": str(game["venue"]["id"]),
                    "venue_name": game["venue"]["name"],
                    "is_home": int(side == "home"),
                    "night_game": int(game.get("dayNight") == "night"),
                    "roster_position": position.get("abbreviation", position.get("name", "")),
                })
            if eligible == 0:
                skipped.append((game["gamePk"], side + ": no active non-pitchers returned"))

    result = pd.DataFrame(rows)
    if not result.empty:
        result["game_date"] = pd.to_datetime(result["game_date"])
        result = result.drop_duplicates(["game_pk", "batter"]).reset_index(drop=True)
    if skipped:
        print(f"Skipped {len(skipped)} early game/team entries; first examples: {skipped[:4]}")
    return result


def add_projected_batting_orders(rows, history, target_date):
    """Estimate lineup position conditional on the rostered hitter starting."""
    rows = rows.copy()
    cutoff = pd.Timestamp(target_date)
    hist = history[pd.to_datetime(history.game_date) < cutoff].copy()
    hist["game_date"] = pd.to_datetime(hist["game_date"])
    hist["batter"] = pd.to_numeric(hist["batter"]).astype("int64")

    recent = hist[hist.game_date >= cutoff - pd.Timedelta(days=60)]
    long = hist[hist.game_date >= cutoff - pd.Timedelta(days=365)]

    def summaries(frame, keys):
        if frame.empty:
            return {}, {}
        grouped = frame.groupby(keys, dropna=False).agg(
            order=("batting_order", "mean"),
            starts=("game_pk", "nunique"),
        )
        return grouped.order.to_dict(), grouped.starts.to_dict()

    hand_60_order, hand_60_n = summaries(recent, ["batter", "pitcher_hand"])
    all_60_order, all_60_n = summaries(recent, ["batter"])
    hand_365_order, hand_365_n = summaries(long, ["batter", "pitcher_hand"])
    all_365_order, all_365_n = summaries(long, ["batter"])

    projected, sources, samples = [], [], []
    for row in rows.itertuples(index=False):
        batter = int(row.batter)
        hand_key = (batter, row.pitcher_hand)
        if hand_60_n.get(hand_key, 0) >= 3:
            order, count, source = hand_60_order[hand_key], hand_60_n[hand_key], "60d vs starter hand"
        elif all_60_n.get(batter, 0) >= 3:
            order, count, source = all_60_order[batter], all_60_n[batter], "60d all starts"
        elif hand_365_n.get(hand_key, 0) >= 3:
            order, count, source = hand_365_order[hand_key], hand_365_n[hand_key], "365d vs starter hand"
        elif all_365_n.get(batter, 0) >= 1:
            order, count, source = all_365_order[batter], all_365_n[batter], "365d all starts"
        else:
            order, count, source = 7.0, 0, "No prior start; default 7th"
        projected.append(float(np.clip(order, 1, 9)))
        sources.append(source)
        samples.append(int(count))

    rows["projected_batting_order"] = np.round(projected, 2)
    # The fitted model expects the original training column name.
    rows["batting_order"] = rows["projected_batting_order"]
    rows["projected_order_source"] = sources
    rows["projected_order_starts"] = samples
    return rows


def run_early_roster(stores, data, choice, target_date=TARGET_DATE):
    """Rank active hitters by P(HR | starts) before lineups are confirmed."""
    expected_end = (pd.Timestamp(target_date) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    if choice.get("history_end") != expected_end:
        raise ValueError("The prediction date changed or history is stale. Rerun run_history for this target date first.")

    schedule = get_schedule(target_date, target_date, historical=False)
    rows = active_roster_rows(schedule, target_date)
    if rows.empty:
        print("No eligible active-roster matchups with a named opposing starter are available yet.")
        return pd.DataFrame()

    people = get_people(set(rows.batter) | set(rows.starter))
    rows = add_handedness(rows, people)
    rows = add_projected_batting_orders(rows, data, target_date)
    future = features(rows, stores)
    model = fit_daily_model(data, choice, target_date)
    result = predict_rankings(future, model)

    result["conditional_hr_chance_pct"] = result["hr_chance_pct"]
    result["probability_basis"] = "P(HR | starts); projected batting order"
    result["generated_at_utc"] = datetime.now(ZoneInfo("UTC")).isoformat()

    filename = f"HR_Early_Roster_Rankings_{target_date}.csv"
    result.to_csv(filename, index=False)
    early_dir = MODEL_DIR / "early_roster_rankings"
    early_dir.mkdir(parents=True, exist_ok=True)
    result.to_csv(early_dir / filename, index=False)

    print(f"Ranked {len(result):,} active non-pitchers across {result.game_pk.nunique()} upcoming games.")
    print("Probabilities are conditional on the hitter starting; no start-probability discount was applied.")
    print("A projected batting order is used until the confirmed lineup is available.")
    return result



## 11. Run the download, feature creation and backtest

This is the first long-running cell. The returned `model_data` has the raw labels and the pregame features, so you can inspect individual rows. **No results have been prefilled.** If the Statcast download is inaccessible, this cell will stop before fitting a model.

In [ ]:
stores, model_data, selected_model, validation_trials, test_metrics, calibration_table, test_predictions = run_history(TARGET_DATE)

display(validation_trials.round(5))
display(test_metrics.round(5))
display(calibration_table.round(4))

model_loss = test_metrics.loc[test_metrics.model.eq("Selected model"), "log_loss"].iloc[0]
baseline_loss = test_metrics.loc[test_metrics.model.eq("Past hitter HR/game"), "log_loss"].iloc[0]
if model_loss >= baseline_loss:
    print("The selected model did not beat the historical hitter-rate baseline on held-out log loss. Treat it as a prototype and investigate before relying on it.")
else:
    print("The selected model beat the historical hitter-rate baseline on held-out log loss. Check calibration and performance over additional future periods before drawing a strong conclusion.")

test_metrics.to_csv("HR_Backtest_Metrics.csv", index=False)
test_predictions.to_csv("HR_Backtest_Predictions.csv", index=False)

## 12. Check calibration visually

Each point is a group of similarly rated hitters. Points near the diagonal indicate that predicted probabilities and actual HR rates agree in that group. The sample count is in the table above; very small groups give noisier comparisons. This plot is generated from actual held-out results after you run the backtest.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(calibration_table.predicted, calibration_table.actual, "o-", label="Held-out games")
limit = min(1, max(calibration_table.predicted.max(), calibration_table.actual.max()) * 1.15)
ax.plot([0, limit], [0, limit], "--", color="gray", label="Predicted = observed")
ax.set(xlabel="Predicted probability of at least one HR", ylabel="Observed fraction with at least one HR", title="Home-run probability calibration", xlim=(0, limit), ylim=(0, limit))
ax.legend()
fig.tight_layout()
plt.show()

## 13. Generate the early active-roster board

Run this before confirmed lineups are posted. It includes every active non-pitcher in an eligible game and ranks the chance of a home run **conditional on that hitter starting**. It deliberately does not reduce the probability for lineup uncertainty.

Because batting order affects plate-appearance opportunity, the notebook estimates a lineup spot from the hitter's prior starts, preferring recent starts against the same pitcher hand. The `projected_order_source` and `projected_order_starts` columns show how that estimate was made. A hitter with no usable starting history defaults to seventh and should be treated cautiously.

The exact opposing probable starter is still required. Refresh the confirmed board later for lineup spots, scratches and starter changes. If your contest voids nonstarters, verify the rule shown for your specific entry; a void can still change the entry size or payout even when it is not graded as a loss.


In [ ]:
early_roster_rankings = run_early_roster(
    stores, model_data, selected_model, TARGET_DATE
)

if not early_roster_rankings.empty:
    early_columns = [
        "rank", "player", "team", "roster_position",
        "opposing_starter", "projected_batting_order",
        "conditional_hr_chance_pct", "projected_order_source",
        "projected_order_starts", "hitter_barrel_pct",
        "recent_barrel_pct", "most_common_pitch",
        "pitch_fit_edge_pp", "history_flag",
    ]
    display(early_roster_rankings[early_columns].head(25).round(2))
    print("CSV saved locally and in Drive:",
          f"HR_Early_Roster_Rankings_{TARGET_DATE}.csv")


## 14. Generate the confirmed-lineup rankings

Run this after lineups are posted. It supersedes the early-board estimate and refresh it for scratches or starter changes. If nothing appears, check whether the games have started or the lineups are still missing.

`hr_chance_pct` is the estimated probability for that game. `pitch_fit_edge_pp` is only a descriptive pitch-fit feature relative to the hitter's broader same-hand contact profile; it is **not** a change in HR probability. The other columns show useful context, not a causal explanation of the fitted prediction. If the baseline specification won, the output still shows matchup context, but those extra features did not drive its probabilities.

`Limited history` flags a hitter or pitcher with fewer than 100 prior-year PAs in the available history. It is not a formal confidence interval. Early-career estimates should be treated as less supported.

In [ ]:
daily_rankings = run_daily(stores, model_data, selected_model, TARGET_DATE)

if not daily_rankings.empty:
    columns = ["rank", "player", "team", "opposing_starter", "batting_order",
               "hr_chance_pct", "hitter_barrel_pct", "recent_barrel_pct",
               "most_common_pitch", "pitch_fit_edge_pp", "history_flag"]
    display(daily_rankings[columns].head(25).round(2))
    print("CSV saved:", f"HR_Rankings_{TARGET_DATE}.csv")
    print("Per-player/day CSV saved:", f"HR_Player_Day_Rankings_{TARGET_DATE}.csv")

## 15. See the confirmed player/day view

On a normal slate, this agrees with the per-game ranking. For a doubleheader, check `games_covered` carefully. It includes only the posted, not-yet-started games that were eligible when the feed was read.

In [ ]:
if not daily_rankings.empty:
    display(player_day_rankings(daily_rankings).head(25).round(2))

In [ ]:
# ============================================================
# Save forward predictions and score yesterday's predictions
# ============================================================

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

FORWARD_DIR = MODEL_DIR / "forward_test"
FORWARD_DIR.mkdir(parents=True, exist_ok=True)


def save_forward_snapshot():
    """Save the current pregame predictions without overwriting earlier ones."""

    if "daily_rankings" not in globals() or daily_rankings.empty:
        print("No rankings are available to save.")
        return

    snapshot_time = datetime.now(
        ZoneInfo("UTC")
    ).strftime("%Y-%m-%dT%H:%M:%SZ")

    filename_time = datetime.now(
        ZoneInfo("UTC")
    ).strftime("%Y%m%dT%H%M%SZ")

    # Save the game-level predictions used for future scoring.
    game_snapshot = daily_rankings.copy()
    game_snapshot["snapshot_utc"] = snapshot_time

    game_file = FORWARD_DIR / (
        f"game_predictions_{TARGET_DATE}_{filename_time}.csv"
    )

    game_snapshot.to_csv(game_file, index=False)

    # Save the cleaner player/day rankings as well.
    player_snapshot = player_day_rankings(game_snapshot)
    player_snapshot["snapshot_utc"] = snapshot_time

    player_file = FORWARD_DIR / (
        f"player_predictions_{TARGET_DATE}_{filename_time}.csv"
    )

    player_snapshot.to_csv(player_file, index=False)

    print(
        f"Saved {len(game_snapshot):,} game-level predictions "
        f"for {TARGET_DATE}."
    )
    print("Snapshot time:", snapshot_time)
    print("Saved in:", FORWARD_DIR)


def score_forward_date(date_text):
    """Compare saved predictions with final Statcast results."""

    prediction_files = sorted(
        FORWARD_DIR.glob(
            f"game_predictions_{date_text}_*.csv"
        )
    )

    if not prediction_files:
        print(f"No saved predictions found for {date_text}.")
        return pd.DataFrame()

    # Combine all pregame snapshots from that date.
    predictions = pd.concat(
        [pd.read_csv(file) for file in prediction_files],
        ignore_index=True
    )

    predictions["snapshot_utc"] = pd.to_datetime(
        predictions["snapshot_utc"],
        utc=True
    )

    predictions["game_pk"] = pd.to_numeric(
        predictions["game_pk"]
    ).astype("int64")

    predictions["batter"] = pd.to_numeric(
        predictions["batter"]
    ).astype("int64")

    # Keep each player's latest saved prediction before the game.
    predictions = (
        predictions
        .sort_values("snapshot_utc")
        .drop_duplicates(
            ["game_pk", "batter"],
            keep="last"
        )
    )

    # Get the final schedule and identify completed games.
    schedule = get_schedule(
        date_text,
        date_text,
        historical=True
    )

    final_game_ids = {
        int(game["gamePk"])
        for game in schedule
        if game.get("status", {}).get(
            "abstractGameState"
        ) == "Final"
    }

    if not final_game_ids:
        print(f"No completed games found for {date_text}.")
        return pd.DataFrame()

    # Download or load the actual plate-appearance results.
    actual_pa, _ = get_statcast(
        date_text,
        date_text,
        schedule
    )

    actual_pa = actual_pa[
        actual_pa["game_pk"].isin(final_game_ids)
    ]

    actual = (
        actual_pa
        .groupby(
            ["game_pk", "batter"],
            as_index=False
        )
        .agg(
            actual_pa=("pa", "sum"),
            actual_hr=("hr", "sum")
        )
    )

    scored = predictions[
        predictions["game_pk"].isin(final_game_ids)
    ].merge(
        actual,
        on=["game_pk", "batter"],
        how="left"
    )

    # Exclude scratches or players who never recorded a PA.
    scored = scored[
        scored["actual_pa"].notna()
    ].copy()

    if scored.empty:
        print(f"No eligible predictions could be scored for {date_text}.")
        return scored

    scored["hit_hr"] = (
        scored["actual_hr"] > 0
    ).astype(int)

    scored["prediction_date"] = date_text

    scored_file = FORWARD_DIR / f"scored_{date_text}.csv"
    scored.to_csv(scored_file, index=False)

    # Combine every completed forward-test date.
    scored_files = sorted(
        FORWARD_DIR.glob("scored_????-??-??.csv")
    )

    cumulative = pd.concat(
        [pd.read_csv(file) for file in scored_files],
        ignore_index=True
    )

    cumulative = cumulative.drop_duplicates(
        ["game_pk", "batter"],
        keep="last"
    )

    probabilities = cumulative[
        "hr_probability"
    ].clip(0.000001, 0.999999)

    outcomes = cumulative["hit_hr"].astype(int)

    top_ten = (
        cumulative
        .sort_values(
            ["prediction_date", "hr_probability"],
            ascending=[True, False]
        )
        .groupby("prediction_date")
        .head(10)
    )

    summary = pd.DataFrame([{
        "dates_scored":
            cumulative["prediction_date"].nunique(),

        "games_scored":
            cumulative["game_pk"].nunique(),

        "hitter_games_scored":
            len(cumulative),

        "mean_prediction":
            probabilities.mean(),

        "actual_hr_rate":
            outcomes.mean(),

        "log_loss":
            log_loss(
                outcomes,
                probabilities,
                labels=[0, 1]
            ),

        "brier_score":
            brier_score_loss(
                outcomes,
                probabilities
            ),

        "average_precision":
            average_precision_score(
                outcomes,
                probabilities
            ) if outcomes.sum() > 0 else np.nan,

        "top10_hr_rate":
            top_ten["hit_hr"].mean()
    }])

    cumulative.to_csv(
        FORWARD_DIR / "forward_scored_all.csv",
        index=False
    )

    summary.to_csv(
        FORWARD_DIR / "forward_summary.csv",
        index=False
    )

    print(
        f"Scored {len(scored):,} predictions "
        f"from {date_text}."
    )

    display(summary.round(5))

    return scored


# Save today's pregame rankings.
save_forward_snapshot()

# Automatically score yesterday if predictions exist.
yesterday = (
    pd.Timestamp(TARGET_DATE) -
    pd.Timedelta(days=1)
).strftime("%Y-%m-%d")

if list(
    FORWARD_DIR.glob(
        f"game_predictions_{yesterday}_*.csv"
    )
):
    scored_predictions = score_forward_date(yesterday)
else:
    print(
        f"No saved forward predictions for {yesterday}. "
        "Tracking begins with today's rankings."
    )


## What I would improve next

1. **Establish useful performance.** Run the full download and held-out evaluation first. Check data coverage and reconcile official game totals. Then record genuinely forward predictions, with the lineup and starter available at that exact time.
2. **Audit whether the matchup information helps.** Compare the baseline, baseline plus recent form, baseline plus pitching/park, and full pitch-fit version on later validation periods. The current notebook compares the baseline to the full package; it does not isolate the contribution of every group.
3. **Add weather and roof status.** Use forecasts available before first pitch and archived forecasts for honest backtests. Do not train on final recorded weather and silently substitute forecasts at prediction time.
4. **Refine the baseball matchup.** Add pull-air-ball tendencies, hitter response by pitch velocity/movement/location, and the probable bullpen's personnel and availability. The current bullpen is a trailing team aggregate, not a projection of which relievers will pitch today.
5. **Test a nonlinear challenger.** Try gradient boosting after the logistic baseline is solid, using the same chronological splits and calibration checks. Keep only changes that improve later-period predictions.

The current version does not explicitly project the number of PAs, starter innings, pinch-hit risk or each reliever faced. Batting order and historical game outcomes partly capture those effects. A more detailed per-PA model could estimate separate starter/bullpen matchups and then aggregate over a distribution of plate appearances. Do not multiply per-PA chances by an actual future PA count in a backtest.

Other practical limits: historical records can be revised, player handedness is read from current profiles, scarce pitch-specific samples remain uncertain after smoothing, and this version excludes postseason games and suspended games. None of the coefficients should be interpreted as causal effects.

The aim is a well-tested ranking of probabilities. Even a correctly estimated top-ranked hitter will often finish the game without a home run.